## Implementation of Query Enhancement - Step-back Prompting

### Libraries, ChatOllama, Chroma vectorstore, LLM prompt initialization

In [1]:
from chromadb.config import Settings
from chromadb import Client
from langchain.vectorstores import Chroma
import chromadb

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from typing_extensions import List, TypedDict
from langgraph.graph import START, StateGraph

import os, re
from datetime import datetime

date = datetime.today().strftime('%Y-%m-%d')

# Initialize Langsmith
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGSMITH_PROJECT"] = f"[{date}] VAA - Query Enhancement (Step-back Prompting)"

# Initialize LLM
REASONING = True

llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, temperature=0.6, reasoning=True if REASONING else False)
emb = OllamaEmbeddings(model="bge-m3:567m")

In [ ]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_eee_document" if not SINGLE else "vaa_documents"

# Initialize retriever for queries. Get the workspace root directory
import pathlib
workspace_root = pathlib.Path(__file__).parent.parent.parent if '__file__' in globals() else pathlib.Path.cwd().parent.parent
chroma_db_path = workspace_root / "chroma_db"

client = Client(Settings())
client = chromadb.PersistentClient(path=str(chroma_db_path))

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=emb
)

client.get_collection(name=collection_name).count()

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_6296/3439006483.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


[Collection(name=vaa_documents)]

In [ ]:
# The LLM prompt
LLM_prompt = \
    """
    You are a professional academic advisor at The Hong Kong Polytechnic University. Please adhere to the following rules:
        1. Answer in the same language as the user query, e.g., English query, English answer.
        2. Avoid saying "may", "maybe", or anything similar; be affirmative, confident, and decisive in your answers.
        3. Avoid saying "based on the provided context", or anything similar; answer directly.
        4. Say no if you cannot answer the question; do not fabricate a factually false answer.
        5. Provide advice to the student if necessary.

    Now, please use the following context to answer the student's question.
    Remember to be nice and ask if there are any more enquiries.

    *Context*:
    ----------
    {context}
    ----------

    *Student's Question*:
    {question}

    Helpful Answer:
    """
prompt = PromptTemplate.from_template(LLM_prompt)

### Step-back prompting implementation

In [ ]:
# Step-back prompt (Source: Langchain)
STEP_BACK_PROMPT = \
    """
    You are an expert at world knowledge. 
    Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:

    Human: Could the members of The Police perform lawful arrests?
    AI: what can the members of The Police do?

    Human: Jan Sindel’s was born in what country? 
    AI: what is Jan Sindel’s personal history?

    *Human*: {question}
    AI:
    """
step_back_prompt = PromptTemplate.from_template(STEP_BACK_PROMPT)

In [ ]:
# Defining the class structure for the LLM
class State(TypedDict):
    question: str
    step_back_question: str
    context: List[Document]
    answer: str

# Function to generate step-back question
def generate_step_back(state: State):
    question = state["question"]
    messages = step_back_prompt.invoke({"question": question})
    response = llm.invoke(messages)
    return {"step_back_question": response.content}

# Functions for document retrieval based on cos-sim
def retrieve(state: State):
    retrieved_docs = vectorStore.similarity_search(state["question"], k=3)
    step_back_docs = vectorStore.similarity_search(state["step_back_question"], k=3)
    
    # Combine and remove duplicated documents
    all_docs = retrieved_docs + step_back_docs
    unique_docs = []
    seen_content = set()
    for doc in all_docs:
        if doc.page_content not in seen_content:
            unique_docs.append(doc)
            seen_content.add(doc.page_content)
            
    return {"context": unique_docs}

# Functions for constructing the final LLM prompt
def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    #print(response.additional_kwargs)

    # Include the reasoning part in the output
    return {"answer": f"<think>\n{response.additional_kwargs.get('reasoning_content', '')}</think>\n\n{response.content}"}

    # NOT Include the reasoning part
    #return {"answer": f"{response.content}"}

# Functions for graph building (a process sequence)
def graph_building():
    global graph
    graph_builder = StateGraph(State).add_sequence([generate_step_back, retrieve, generate])
    graph_builder.add_edge(START, "generate_step_back")
    graph = graph_builder.compile()

graph_building()

### Optional: Testing

In [ ]:
query = \
"What is the potential career path for studying in BEng Scheme in IAIE?"

print(f"Generating {query}")
result = graph.invoke({"question": query})

print(f"\nStep-back Question generated:\n")
for q in result['step_back_question']:
    print(f"- {q}") 
print(f"\nPlease see Langsmith for the full details of the execution trace.\n")